In [49]:
import os
import json

INPUT_DIR = r'../carga_didatica/2026-2/formularios/'

In [50]:
resultado = {"versao": "3.0", "formularios": {}, "disciplinas": {}, "docentes": [], "saldos": {}}

# Disciplinas

In [51]:
disciplinas: list[dict] = json.load(open('public/turmas.json', 'r',  encoding='utf-8'))

In [52]:
for d in disciplinas:
    d['cursos'] = d['curso']
    d['docentes'] = [] # Aproveitando que já está sendo iterado
    del d['curso']

## Ajustar as turmas adicionando o campo "docentes"

In [53]:
dictPlinas = {disciplinas[i]['id'] : disciplinas[i] for i in range(0,len(disciplinas))}

# Turmas espelho: a turma da pós foi mesclada na turma da graduação.
# Caso algum formulário ainda traga o id da pós, ele é convertido para o id da graduação.
aliasEspelho = {
    d['espelho']['id']: d['id']
    for d in disciplinas
    if isinstance(d.get('espelho'), dict)
}

# Formulários

In [54]:
docentesAvaliacao = {}

In [55]:
# # Percorre todos os arquivos .json da pasta
# for arquivo in os.listdir(INPUT_DIR):
#     if arquivo.endswith('.json'):
#         caminho = os.path.join(INPUT_DIR, arquivo)
#         with open(caminho, 'r', encoding='utf-8') as f:
#             dados = json.load(f)
#             nome_arquivo = caminho.split('/')[-1].split('.json')[0]
#             for nome, info in dados.items():
#                 formulariosDocente = {}

#                 for key, value in info.get("formularios", {}).items():
#                     disciplina, turma = key.split(',')

#                     if(key in dictPlinas.keys()):
#                         formulariosDocente[key] = value
#                     else:
#                         print(info)
                    
#                 resultado["formularios"][nome_arquivo] = formulariosDocente

# Percorre todos os arquivos .json da pasta
for arquivo in os.listdir(INPUT_DIR):
    if arquivo.endswith('.json'):
        caminho = os.path.join(INPUT_DIR, arquivo)
        with open(caminho, 'r', encoding='utf-8') as f:
            dados = json.load(f)
            nome_arquivo = os.path.splitext(os.path.basename(caminho))[0] 
            
            docentesAvaliacao[nome_arquivo] = dados[list(dados.keys())[0]]['avaliacao']

            for nome, info in dados.items():
                formulariosDocente = {}

                for key, value in info.get("formularios", {}).items():
                    
                    key_normalizada = key # Assume a chave original como padrão
                    try:
                        # 1. Tenta separar a disciplina e a turma
                        disciplina, turma_str = key.split(',')
                        
                        # 2. Normaliza a turma:
                        # '1.0' -> float(1.0) -> int(1) -> str('1')
                        # '1'   -> float(1.0) -> int(1) -> str('1')
                        turma_normalizada_str = str(int(float(turma_str)))
                        
                        # 3. Remonta a chave normalizada
                        key_normalizada = f"{disciplina},{turma_normalizada_str}"
                        
                    except ValueError:
                        pass # Deixa a key_normalizada como a original se falhar
                    
                    # Converte o id da turma da pós para o id da turma espelho da graduação
                    if key_normalizada in aliasEspelho:
                        print(f"{nome_arquivo}: '{key_normalizada}' é espelho -> '{aliasEspelho[key_normalizada]}'")
                        key_normalizada = aliasEspelho[key_normalizada]

                    # Checa a CHAVE NORMALIZADA contra o DICT
                    if(key_normalizada in dictPlinas):
                        # Salva a CHAVE NORMALIZADA no resultado
                        formulariosDocente[key_normalizada] = value
                        
                    else:
                        if(key_normalizada+'1' in dictPlinas):
                            # Agora salvamos a CHAVE NORMALIZADA no resultado
                            formulariosDocente[key_normalizada+'1'] = value
                        else:
                            print(f"Docente: {nome}, Chave não encontrada: {key} (Normalizada: {key_normalizada})")
                
                resultado["formularios"][nome_arquivo] = formulariosDocente

Docente: Afonso Paiva Neto, Chave não encontrada: SME0250,1 (Normalizada: SME0250,1)
Docente: Antonio Castelo Filho, Chave não encontrada: SME0250,1 (Normalizada: SME0250,1)
Docente: Antonio Castelo Filho, Chave não encontrada: SME5827,1 (Normalizada: SME5827,1)
Docente: Antonio Castelo Filho, Chave não encontrada: MAI5002,1 (Normalizada: MAI5002,1)
Docente: Fabrício Simeoni de Sousa, Chave não encontrada: SME0252,1 (Normalizada: SME0252,1)
Docente: Fabrício Simeoni de Sousa, Chave não encontrada: SME0250,1 (Normalizada: SME0250,1)
Docente: Leandro Franco de Souza, Chave não encontrada: SME5873,1 (Normalizada: SME5873,1)
Docente: Leandro Franco de Souza, Chave não encontrada: SME0252,1 (Normalizada: SME0252,1)
Docente: Marinho Gomes de Andrade Filho, Chave não encontrada: SME0870,1 (Normalizada: SME0870,1)
Docente: Miguel Vinicius Santini Frasson, Chave não encontrada: PMA5641,1 (Normalizada: PMA5641,1)
Docente: Miguel Vinicius Santini Frasson, Chave não encontrada: PRG0027,1 (Normaliz

In [56]:
for docente, turmasDocente in resultado.get("formularios", {}).items():
    if not turmasDocente:
        continue

    for turma in turmasDocente.keys():
        tDict = dictPlinas.get(turma)
        if tDict is not None:
            tDict.setdefault('docentes', []).append(docente)

resultado["disciplinas"] = disciplinas

# Docentes e Saldos

In [57]:
import pandas as pd

In [58]:
# df = pd.read_excel(INPUT_DIR + "/_ICMC _listagem docentes-somente docentes SME.xlsx").dropna( subset = [ 'Carga' ] )
df = pd.read_excel("../carga_didatica/2026-2/TodoSME_Tabela_CargaDidatica_2026-1S.xlsx").dropna( subset = [ 'Carga' ] )

In [59]:
df

,Docente,Saldo Anterior,Gestão,Bônus Gestão,Carga,Carga Gestão,Status,Carga Ponderada,Carga Gestão Ponderada,Novo Saldo
0,Adriano Kamimura Suzuki,2.259096,PIPGEs,0.25,3.100,0.442399,1.0,3.100,0.442399,3.804722
1,Afonso Paiva Neto,-0.020904,MECAI,0.25,1.550,0.442399,1.0,1.550,0.442399,-0.025278
2,André Carlos Ponce de Leon Ferreira de Carvalho,5.701923,ICMC,1.00,0.500,1.769595,1.0,0.500,1.769595,5.974745
3,Antonio Castelo Filho,1.179096,CPG,0.25,1.000,0.442399,1.0,1.000,0.442399,0.624722
4,Cibele Maria Russo Noveli,2.139096,PRIP,0.25,2.000,0.442399,1.0,2.000,0.442399,2.584722
5,Cynthia de Oliveira Lage Ferreira,-0.335180,NaN,NaN,2.000,0.000000,1.0,2.000,0.000000,-0.331953
6,Daiane de Souza Santos,0.394820,NaN,NaN,2.000,0.000000,1.0,2.000,0.000000,0.398047
7,Eduardo Fontoura Costa,-0.965180,NaN,NaN,2.000,0.000000,1.0,2.000,0.000000,-0.961953
8,Eeltje Cornelis Nijholt,-0.776379,NaN,NaN,1.500,0.000000,1.0,1.500,0.000000,-1.273152
9,Elias Salomão Helou Neto,1.009096,BMACC,0.25,2.000,0.442399,1.0,2.000,0.442399,1.454722


In [60]:
df.drop( columns = [ 'Saldo Anterior', 'Gestão', 'Bônus Gestão', 'Carga', 'Carga Gestão', 'Status', 'Carga Ponderada', 'Carga Gestão Ponderada' ])

,Docente,Novo Saldo
0,Adriano Kamimura Suzuki,3.804722
1,Afonso Paiva Neto,-0.025278
2,André Carlos Ponce de Leon Ferreira de Carvalho,5.974745
3,Antonio Castelo Filho,0.624722
4,Cibele Maria Russo Noveli,2.584722
5,Cynthia de Oliveira Lage Ferreira,-0.331953
6,Daiane de Souza Santos,0.398047
7,Eduardo Fontoura Costa,-0.961953
8,Eeltje Cornelis Nijholt,-1.273152
9,Elias Salomão Helou Neto,1.454722


In [61]:
for nome, saldo in zip(df['Docente'], df['Novo Saldo']):
    
    #    Se o 'nome' não estiver no dicionário, retorna um dicionário vazio {}
    avaliacao_docente = docentesAvaliacao.get(nome, {})

    #    Se a chave ('comentario' ou 'preferencia') não existir, retorna ""
    comentario = avaliacao_docente.get('comentario', "")
    preferencia = avaliacao_docente.get('preferencia', "")
    
    # Adiciona os dados à lista "docentes"
    resultado["docentes"].append({
        'nome': nome, 
        "ativo": True, 
        "comentario": comentario, 
        "agrupar": preferencia
    })
    
    # Atualiza o dicionário "saldos". 
    resultado["saldos"][nome] = saldo

Preenche a tag `formulario` com `{}` caso o docente não tenha aprentado formulários.

In [62]:
docentes_nao_encontrados = []

for nome in df['Docente']:
    if not resultado['formularios'].get(nome):
        docentes_nao_encontrados.append(nome)
        resultado['formularios'][nome] = {}

In [63]:
print("==============================================================================")
print(' Docentes que estão na planilha mas que não foram encontrados nos formulários')
print("==============================================================================")
for nome in docentes_nao_encontrados:
    print(nome)
print('-'*20)
print(f'Total: {len(docentes_nao_encontrados)}')
print("==============================================================================")
print("Veirficar os nomes dos arquivos .json como também o nome inserido dentro deles.")


 Docentes que estão na planilha mas que não foram encontrados nos formulários
André Carlos Ponce de Leon Ferreira de Carvalho
Cibele Maria Russo Noveli
Eeltje Cornelis Nijholt
--------------------
Total: 3
Veirficar os nomes dos arquivos .json como também o nome inserido dentro deles.


André Carlos Ponce de Leon Ferreira de Carvalho <br/>
Antonio Castelo Filho <br/>
Daniel Henrique Silva <br/>
Eeltje Cornelis Nijholt <br/>
Jorge Luis Bazan Guzman <br/>
José Alberto Cuminato <br/>
Leandro Franco de Souza <br/>
Marinho Gomes de Andrade Filho <br/>
Miguel Vinicius Santini Frasson <br/>
Tiago Pereira da Silva <br/>

In [ ]:
# Se quiser salvar esse resultado em um novo arquivo:
with open('formularios_combinados.json', 'w', encoding='utf-8') as f:
    json.dump(resultado, f, ensure_ascii=False, indent=2)

: 